### Siying Li
### Take_Home exercise3
### instructions: For this assignment, we’d like you to use the F1 Datasets we have been using for the class to build any ML model of your choice and track the model for each run using MLflow. Select any of the F1 datasets in AWS S3 to build your model. You are allowed to join multiple datasets.



In [0]:
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True, inferSchema=True)
display(df_results)

In [0]:
df_qualifying = spark.read.csv("/Volumes/gr5069/raw/f1_data/qualifying.csv", header=True, inferSchema=True)
display(df_qualifying)

In [0]:
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True, inferSchema=True)
display(df_races)

In [0]:
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True, inferSchema=True)
display(df_drivers)

In [0]:
from pyspark.sql.functions import col, when, datediff, lit

# 1. Select needed columns

results_sel = (
    df_results
    .select(
        "raceId",
        "driverId",
        "constructorId",
        "grid",
        "positionOrder"
    )
)

quali_sel = (
    df_qualifying
    .select(
        "raceId",
        "driverId",
        col("position").alias("quali_position")
    )
)

races_sel = (
    df_races
    .select(
        "raceId",
        "year",
        "round",
        "circuitId",
        "date"
    )
)

drivers_sel = (
    df_drivers
    .select(
        "driverId",
        "dob"
    )
)

# 2. Join tables

df_model = (
    results_sel
    .join(quali_sel, on=["raceId", "driverId"], how="left")
    .join(races_sel, on="raceId", how="left")
    .join(drivers_sel, on="driverId", how="left")
)

# 3. Create target variable

df_model = df_model.withColumn(
    "podium_flag",
    when(col("positionOrder") <= 3, 1).otherwise(0)
)

# 4. Create driver age

df_model = df_model.withColumn(
    "driver_age",
    datediff(col("date"), col("dob")) / lit(365.25)
)

# 5. Keep final columns

df_model = (
    df_model
    .select(
        "raceId",
        "driverId",
        "constructorId",
        "grid",
        "quali_position",
        "year",
        "round",
        "circuitId",
        "driver_age",
        "podium_flag"
    )
)

display(df_model)

In [0]:
from pyspark.sql.functions import count, sum, isnan

# 1. Row count
print("Number of rows in df_model:", df_model.count())

# 2. Check missing values in key columns
df_model.select([
    count(when(col(c).isNull(), c)).alias(c + "_missing")
    for c in df_model.columns
]).show()

# 3. Check label distribution
df_model.groupBy("podium_flag").count().orderBy("podium_flag").show()

# 4. Basic summary statistics
display(df_model.summary())

In [0]:
from pyspark.sql.functions import col

df_model_v1 = (
    df_model
    .select(
        "raceId",
        "driverId",
        "constructorId",
        "grid",
        "year",
        "round",
        "circuitId",
        "driver_age",
        "podium_flag"
    )
    .dropna(
        subset=[
            "constructorId",
            "grid",
            "year",
            "round",
            "circuitId",
            "driver_age",
            "podium_flag"
        ]
    )
)

print("Rows in df_model_v1:", df_model_v1.count())
display(df_model_v1)

In [0]:
from pyspark.sql.functions import count, when

df_model_v1.select([
    count(when(col(c).isNull(), c)).alias(c + "_missing")
    for c in df_model_v1.columns
]).show()

df_model_v1.groupBy("podium_flag").count().orderBy("podium_flag").show()

df_model_v1.printSchema()
display(df_model_v1.summary())

In [0]:
import pandas as pd

df_pd = df_model_v1.toPandas()

print(df_pd.shape)
df_pd.head()

In [0]:
X = df_pd[[
    "constructorId",
    "grid",
    "year",
    "round",
    "circuitId",
    "driver_age"
]]

y = df_pd["podium_flag"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nX preview:")
print(X.head())

print("\ny preview:")
print(y.head())

In [0]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTraining label distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting label distribution:")
print(y_test.value_counts(normalize=True))

In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=4,
    random_state=42
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=6,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

In [0]:
import pandas as pd
from sklearn.metrics import confusion_matrix

# 1. confusion matrix
cm = confusion_matrix(y_test, y_pred)
confusion_df = pd.DataFrame(
    cm,
    index=["actual_0", "actual_1"],
    columns=["pred_0", "pred_1"]
)

print("Confusion Matrix:")
print(confusion_df)

# 2. feature importance
feature_importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

print("\nFeature Importance:")
print(feature_importance_df)

The balanced Random Forest model performs much better than the initial baseline because it is able to identify the minority class more effectively. While precision is still limited, recall improves substantially, which means the model captures most true podium finishes. The confusion matrix also shows that the model still over-predicts podium outcomes, but this trade-off is acceptable at this stage because the earlier version failed to detect most podium cases at all. In addition, feature importance results indicate that grid position is the dominant predictor, followed by constructor ID, which is consistent with expectations about race performance in Formula 1.

In [0]:
import mlflow
import mlflow.sklearn
import tempfile
import os

with mlflow.start_run(run_name="RF_baseline_podium_balanced") as run:
    
    # 1. train model
    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=6,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
    
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    # 2. metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # 3. log model
    mlflow.sklearn.log_model(rf, "random-forest-model")

    # 4. log params
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("random_state", 42)

    # 5. log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1", f1)

    # 6. feature importance
    feature_importance_df = pd.DataFrame({
        "feature": X.columns,
        "importance": rf.feature_importances_
    }).sort_values("importance", ascending=False)

    # 7. confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    confusion_df = pd.DataFrame(
        cm,
        index=["actual_0", "actual_1"],
        columns=["pred_0", "pred_1"]
    )

    # 8. save artifacts
    temp = tempfile.NamedTemporaryFile(prefix="feature-importance-", suffix=".csv", delete=False)
    feature_path = temp.name
    temp.close()

    temp = tempfile.NamedTemporaryFile(prefix="confusion-matrix-", suffix=".csv", delete=False)
    confusion_path = temp.name
    temp.close()

    try:
        feature_importance_df.to_csv(feature_path, index=False)
        confusion_df.to_csv(confusion_path)

        mlflow.log_artifact(feature_path, "feature-importance")
        mlflow.log_artifact(confusion_path, "confusion-matrix")
    finally:
        if os.path.exists(feature_path):
            os.remove(feature_path)
        if os.path.exists(confusion_path):
            os.remove(confusion_path)

    print("Run ID:", run.info.run_id)
    print("Experiment ID:", run.info.experiment_id)
    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1:", f1)

In [0]:
param_grid = [
    {"n_estimators": 50,  "max_depth": 4,  "class_weight": "balanced", "random_state": 42},
    {"n_estimators": 100, "max_depth": 4,  "class_weight": "balanced", "random_state": 42},
    {"n_estimators": 200, "max_depth": 4,  "class_weight": "balanced", "random_state": 42},
    {"n_estimators": 50,  "max_depth": 6,  "class_weight": "balanced", "random_state": 42},
    {"n_estimators": 100, "max_depth": 6,  "class_weight": "balanced", "random_state": 42},
    {"n_estimators": 200, "max_depth": 6,  "class_weight": "balanced", "random_state": 42},
    {"n_estimators": 50,  "max_depth": 8,  "class_weight": "balanced", "random_state": 42},
    {"n_estimators": 100, "max_depth": 8,  "class_weight": "balanced", "random_state": 42},
    {"n_estimators": 200, "max_depth": 8,  "class_weight": "balanced", "random_state": 42},
    {"n_estimators": 100, "max_depth": 10, "class_weight": "balanced", "random_state": 42}
]

print("Number of runs:", len(param_grid))

In [0]:
import mlflow
import mlflow.sklearn
import tempfile
import os
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

all_results = []

for i, params in enumerate(param_grid, start=1):
    with mlflow.start_run(run_name=f"RF_run_{i}") as run:
        
        # 1. train model
        rf = RandomForestClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            class_weight=params["class_weight"],
            random_state=params["random_state"],
            n_jobs=-1
        )
        
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_test)

        # 2. metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)

        # 3. log model
        mlflow.sklearn.log_model(rf, name="random-forest-model")

        # 4. log params
        for k, v in params.items():
            mlflow.log_param(k, v)

        # 5. log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1", f1)

        # 6. feature importance
        feature_importance_df = pd.DataFrame({
            "feature": X.columns,
            "importance": rf.feature_importances_
        }).sort_values("importance", ascending=False)

        # 7. confusion matrix
        cm = confusion_matrix(y_test, y_pred)
        confusion_df = pd.DataFrame(
            cm,
            index=["actual_0", "actual_1"],
            columns=["pred_0", "pred_1"]
        )

        # 8. log artifacts
        temp = tempfile.NamedTemporaryFile(prefix=f"feature-importance-run{i}-", suffix=".csv", delete=False)
        feature_path = temp.name
        temp.close()

        temp = tempfile.NamedTemporaryFile(prefix=f"confusion-matrix-run{i}-", suffix=".csv", delete=False)
        confusion_path = temp.name
        temp.close()

        try:
            feature_importance_df.to_csv(feature_path, index=False)
            confusion_df.to_csv(confusion_path)

            mlflow.log_artifact(feature_path, "feature-importance")
            mlflow.log_artifact(confusion_path, "confusion-matrix")
        finally:
            if os.path.exists(feature_path):
                os.remove(feature_path)
            if os.path.exists(confusion_path):
                os.remove(confusion_path)

        # 9. save results locally for comparison
        all_results.append({
            "run_name": f"RF_run_{i}",
            "n_estimators": params["n_estimators"],
            "max_depth": params["max_depth"],
            "class_weight": params["class_weight"],
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "run_id": run.info.run_id
        })

        print(f"Completed RF_run_{i} | accuracy={accuracy:.4f}, precision={precision:.4f}, recall={recall:.4f}, f1={f1:.4f}")

### Best Model Explanation
I chose RF_run_10 as my best model because it gave the strongest overall performance across the ten experiments, especially when I focused on F1 score rather than accuracy alone. Since podium finishes are the minority class in my dataset, I thought F1 was a more meaningful metric because it reflects the balance between precision and recall. Although some other runs had slightly higher recall, RF_run_10 had the highest F1, the highest precision, and also the highest accuracy, while still keeping recall at a reasonably strong level. In other words, this run did the best job of balancing two goals at the same time: identifying true podium finishers and avoiding too many false positives. The parameter setting for this model was n_estimators = 100, max_depth = 10, class_weight = balanced, and random_state = 42, and overall this combination seemed to work best for handling the class imbalance and capturing useful patterns in the data.